# CLIP: совместить изображения и подписи

**Learning Transferable Visual Models From Natural Language Supervision** · Radford et al. · 2021

[Оригинальная статья](https://arxiv.org/abs/2103.00020)

**Цель урока:** Обучить два encoder с симметричным contrastive loss и проверить retrieval и перепутанные пары.

## Перед началом

Softmax, cross entropy, базовый урок VLM.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

CLIP учится сопоставлять изображение с текстовым описанием, а не генерировать описание токен за токеном. Два encoder дают векторы в одном пространстве; правильные пары сближаются относительно других пар batch.

Отрицательные примеры здесь — остальные элементы batch. Если разные подписи обозначают одно и то же изображение, обычный single-positive loss может считать их ложными отрицательными. В маленьком эксперименте используем уникальные синтетические пары. Это позволяет изучить objective, не скачивая pretrained CLIP.

## Граница воспроизведения

Реализован CLIP-style objective на синтетических признаках. Нет image/text towers, web-scale data или zero-shot benchmark.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`u = normalize(image_encoder(x))`, `v = normalize(text_encoder(t))`. `S = exp(log_scale) * u @ v.T`.

`L = (CE(S, arange(B)) + CE(S.T, arange(B))) / 2`. Диагональ содержит совпадающие пары; строки — image→text, столбцы — text→image. Нормализация отделяет направление вектора от его длины. Temperature управляет резкостью softmax, но не заменяет хорошие representations.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Делаем две модальности из общего скрытого сигнала. Linear encoder заменяют большие vision/text towers.

In [ ]:
batch = 24
latent = torch.randn(batch, 6)
image_features = latent @ torch.randn(6, 12) + 0.01 * torch.randn(batch, 12)
text_features = latent @ torch.randn(6, 10) + 0.01 * torch.randn(batch, 10)
image_encoder = nn.Linear(12, 16); text_encoder = nn.Linear(10, 16)
log_scale = nn.Parameter(torch.tensor(2.0))
labels = torch.arange(batch)
def similarities():
    u = F.normalize(image_encoder(image_features), dim=-1)
    v = F.normalize(text_encoder(text_features), dim=-1)
    return log_scale.exp().clamp(max=100) * (u @ v.T)
scores = similarities()
print('similarity shape:', scores.shape)
assert scores.shape == (batch, batch)

## Шаг 2

Обновляем оба encoder. Retrieval accuracy здесь относится только к memorized batch.

In [ ]:
optimizer = torch.optim.AdamW([*image_encoder.parameters(), *text_encoder.parameters(), log_scale], lr=0.02)
history = []
for step in range(120):
    scores = similarities()
    loss = (F.cross_entropy(scores, labels) + F.cross_entropy(scores.T, labels)) / 2
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
    history.append(loss.item())
with torch.no_grad():
    scores = similarities()
    accuracy = (scores.argmax(-1) == labels).float().mean().item()
print('training-batch retrieval:', accuracy)
assert history[-1] < history[0]
plt.imshow(scores.detach(), cmap='Reds'); plt.xlabel('text index'); plt.ylabel('image index'); plt.colorbar(); plt.show()

## Шаг 3

Переставляем подписи, но оставляем старые labels. Это контролируемая проверка того, что loss зависит от pairing.

In [ ]:
permutation = torch.roll(torch.arange(batch), 1)
aligned_loss = F.cross_entropy(scores, labels)
wrong_pair_loss = F.cross_entropy(scores[:, permutation], labels)
print('aligned loss:', aligned_loss.item(), 'wrong-pair loss:', wrong_pair_loss.item())
assert wrong_pair_loss > aligned_loss
# Correct labels restore equivalence after a column permutation.
inverse = torch.argsort(permutation)
corrected = F.cross_entropy(scores[:, permutation], inverse)
assert torch.allclose(corrected, aligned_loss, atol=1e-6)

## Как читать результат

Диагональная структура показывает выученное соответствие обучающих пар. Она не доказывает zero-shot качество на новых изображениях. Перестановка и обратная перестановка labels помогают найти ошибки индексации.

## Самостоятельные эксперименты

1. Уберите normalize и следите за нормами encoder outputs.
2. Продублируйте одну пару: что будет с single-positive labels?
3. Сделайте отдельный test latent и оцените retrieval.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Без normalize модель может менять scores через длину векторов. Дубликаты создают неоднозначные positives и требуют другого target/loss. Test должен быть с новыми latent при тех же генераторах модальностей; иначе меняется сама задача.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.